In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn

import torch.functional as F

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset, Dataset
import torchmetrics

from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

# Подготовка датасета

In [2]:
class SpecDataset(Dataset):

    def __init__(self, path):
        self.path = path
        self.file = h5py.File(self.path, 'r')

    def __len__(self):
        return len(self.file['spec'])

    def __getitem__(self, idx):

        spec = self.file['spec'][idx]
        label = self.file['class_mood_int'][idx]
        label = torch.tensor(label)

        spec = torch.tensor(spec)
        spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    
        return spec, label

In [3]:
dataset = SpecDataset('/kaggle/input/datasets/alexandra841/spec-features/features_resize.h5')

In [4]:
labels = dataset[:][1]

In [5]:
torch.unique(labels, return_counts=True)

(tensor([0, 1, 2, 3, 4, 5], dtype=torch.int8),
 tensor([4326, 1987, 5126, 2452,  793,  919]))

In [6]:
train_val_idx, test_idx = train_test_split(
    np.arange(len(dataset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels
)

trainvalset = torch.utils.data.Subset(dataset, train_val_idx)
testset = torch.utils.data.Subset(dataset, test_idx)

In [7]:
labels_train = labels[train_val_idx]

In [8]:
torch.unique(labels_train, return_counts=True)

(tensor([0, 1, 2, 3, 4, 5], dtype=torch.int8),
 tensor([3461, 1590, 4101, 1961,  634,  735]))

In [9]:
train_idx, val_idx = train_test_split(
    np.arange(len(trainvalset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels_train
)

trainset = torch.utils.data.Subset(trainvalset, train_idx)
valset = torch.utils.data.Subset(trainvalset, val_idx)

In [10]:
train_loader = DataLoader(trainset, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(valset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(testset, batch_size=16, shuffle=False, num_workers=2)

In [11]:
n_classes = 6
in_channels = 1

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Модель 1

## Архитектура

In [26]:
# class CNNModel(nn.Module):

#     def __init__(self):
#         super().__init__()

#         self.conv1 = nn.Conv2d(
#             in_channels = in_channels,
#             out_channels = 8,
#             kernel_size = 5,
#             padding = 'same'
#         )
    
#         self.bn1 = nn.BatchNorm2d(8)
#         self.relu1 = nn.LeakyReLU()
#         self.max_pool1 = nn.MaxPool2d(kernel_size=2)
        
#         self.conv2 = nn.Conv2d(
#             in_channels = 8,
#             out_channels = 16,
#             kernel_size = 5,
#         )
    
#         self.bn2 = nn.BatchNorm2d(16)
#         self.relu2 = nn.LeakyReLU()
#         self.max_pool2 = nn.MaxPool2d(kernel_size=2)
    
#         self.flatten = nn.Flatten()
#         #self.gap = nn.AdaptiveAvgPool2d((1, 1))
#         #self.fc1 = nn.Linear(16, 64)
        
#         self.fc1 = nn.Linear(16 * 30 * 62, 256)
#         self.bn3 = nn.BatchNorm1d(256)
#         self.relu3 = nn.LeakyReLU()
#         self.dropout = nn.Dropout()
#         self.fc2 = nn.Linear(256, n_classes)
    
#     def forward(self, x):
#         x = self.conv1(x)
#         x = self.bn1(x)
#         x = self.relu1(x)
#         x = self.max_pool1(x)
        
#         x = self.conv2(x)
#         x = self.bn2(x)
#         x = self.relu2(x)
#         x = self.max_pool2(x)
#         #x = self.gap(x)         
#         #x = torch.flatten(x, 1)   # (B, 16)

#         x = self.flatten(x)
#         x = self.fc1(x)
#         x = self.bn3(x)
#         x = self.relu3(x)
#         x = self.dropout(x)
#         x = self.fc2(x)
        
#         return x

In [23]:
class CNNModel(nn.Module):
    
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 8, kernel_size=5, padding='same')
        self.bn1 = nn.BatchNorm2d(8)
        self.relu1 = nn.LeakyReLU()
        self.max_pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.relu2 = nn.LeakyReLU()
        self.max_pool2 = nn.MaxPool2d(2)


        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(16, n_classes)

    def forward(self, x):
        x = self.max_pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.max_pool2(self.relu2(self.bn2(self.conv2(x))))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [24]:
model = CNNModel()
model = model.float()

In [25]:
model = model.to(device)

In [17]:
class_weights = compute_class_weight(
        'balanced',
        classes=np.arange(n_classes),
        y=labels_train.numpy()
    )

In [18]:
class_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32
).to(device)

In [19]:
class_weights_tensor

tensor([0.6011, 1.3084, 0.5073, 1.0609, 3.2813, 2.8304], device='cuda:0')

## Обучение

In [20]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor).to(device)

In [21]:
def train_epoch(model, optimizer, train_loader, loss_fn, n_classes):

    loss_log = []
    acc_log = []
    f1_log = []
    
    model.train()

    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    for spec, label in train_loader:
        spec = spec.to(device).float()
        label = label.to(device).long()

        optimizer.zero_grad()
        output_log = model(spec)
        output = torch.argmax(output_log, dim=1)

        loss = loss_fn(output_log, label)
        loss_log.append(loss.item())
        
        #acc = acc_calc(output, label)
        acc_calc.update(output, label)

        #f1 = f1_calc(output, label)
        f1_calc.update(output, label)

        loss.backward()
        optimizer.step()

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1

def test(model, loader, loss_fn, n_classes):
    
    loss_log = []
    acc_log = []
    f1_log = []
    
    model.eval()
    
    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    with torch.no_grad():
        for spec, label in loader:
            spec = spec.to(device).float()
            label = label.to(device).long()
            output_log = model(spec)
            output = torch.argmax(output_log, dim=1)
    
            loss = loss_fn(output_log, label)
            loss_log.append(loss.item())

            #acc = acc_calc(output, label)
            acc_calc.update(output, label)
    
            #f1 = f1_calc(output, label)
            f1_calc.update(output, label)

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1

def train(model, optimizer, n_epoch, train_loader, val_loader, loss_fn, scheduler=None, n_classes=6):
    best_f1 = 0
    
    train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = [], [], [], [], [], []

    for epoch in tqdm(range(n_epoch)):
        train_loss, train_acc, train_f1 = train_epoch(model, optimizer, train_loader, loss_fn, n_classes)
        val_loss, val_acc, val_f1 = test(model, val_loader, loss_fn, n_classes)
        
        train_loss_log.append(train_loss)
        train_acc_log.append(train_acc)
        train_f1_log.append(train_f1)

        val_loss_log.append(val_loss)
        val_acc_log.append(val_acc)
        val_f1_log.append(val_f1)

        print(f"Epoch {epoch}")
        print(f" train loss: {train_loss}, train acc: {train_acc}, train f1: {train_f1}")
        print(f" val loss: {val_loss}, val acc: {val_acc}, val f1: {val_f1}\n")

        if scheduler is not None:
            if isinstance(scheduler,torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_f1)
            else:
                scheduler.step()


        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_model.pt")

    return train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log

In [26]:
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.01, lr=2e-4)

In [27]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.2, patience=1
)

In [28]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model,
    optimizer,
    10,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

 10%|█         | 1/10 [02:49<25:28, 169.86s/it]

Epoch 0
 train loss: 1.7846286656000676, train acc: 0.21003605425357819, train f1: 0.13126802444458008
 val loss: 1.746969939796788, val acc: 0.2755306363105774, val f1: 0.1667555570602417



 20%|██        | 2/10 [05:37<22:30, 168.76s/it]

Epoch 1
 train loss: 1.7427753817576628, train acc: 0.34815704822540283, train f1: 0.22375771403312683
 val loss: 1.734597307101936, val acc: 0.3800560534000397, val f1: 0.2119981348514557



 30%|███       | 3/10 [08:25<19:38, 168.37s/it]

...

Epoch 7
 train loss: 1.714479220792269, train acc: 0.37750402092933655, train f1: 0.24414566159248352
 val loss: 1.7137093263067258, val acc: 0.38165798783302307, val f1: 0.23654383420944214



 90%|█████████ | 9/10 [24:53<02:44, 164.11s/it]

Epoch 8
 train loss: 1.713894682434889, train acc: 0.38251203298568726, train f1: 0.24721819162368774
 val loss: 1.7149387954906294, val acc: 0.38165798783302307, val f1: 0.2489331215620041



100%|██████████| 10/10 [27:42<00:00, 166.20s/it]

Epoch 9
 train loss: 1.7143242072600584, train acc: 0.37790465354919434, train f1: 0.24409565329551697
 val loss: 1.7146524213681555, val acc: 0.3740488588809967, val f1: 0.2475365698337555



# Модель 2

In [55]:
class CNNModel2(nn.Module):
    
    def __init__(self):
        super().__init__()   

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.SiLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2)
        )

        self.gap = nn.AdaptiveAvgPool2d((4, 4))

        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [60]:
model2 = CNNModel2()
model2 = model2.float()
model2 = model2.to(device)

In [61]:
optimizer = torch.optim.AdamW(model2.parameters(), lr=3e-4, weight_decay=1e-4)

In [62]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

In [63]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model2,
    optimizer,
    10,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

 10%|█         | 1/10 [02:57<26:38, 177.56s/it]

Epoch 0
 train loss: 1.7433260419429877, train acc: 0.27423879504203796, train f1: 0.22638478875160217
 val loss: 1.7179384079708415, val acc: 0.2494993954896927, val f1: 0.21073102951049805



 20%|██        | 2/10 [05:59<24:00, 180.09s/it]

Epoch 1
 train loss: 1.6981316562264392, train acc: 0.3063902258872986, train f1: 0.2585463523864746
 val loss: 1.6877578679163745, val acc: 0.30036044120788574, val f1: 0.24308785796165466



 30%|███       | 3/10 [08:56<20:51, 178.83s/it]

...

Epoch 7
 train loss: 1.5828044330462432, train acc: 0.3521634638309479, train f1: 0.3165517747402191
 val loss: 1.6663997864267628, val acc: 0.3520224392414093, val f1: 0.2839798927307129



 90%|█████████ | 9/10 [26:13<02:54, 174.38s/it]

Epoch 8
 train loss: 1.5673265325335355, train acc: 0.3560697138309479, train f1: 0.321953684091568
 val loss: 1.6468309467765176, val acc: 0.3103724420070648, val f1: 0.2846684455871582



100%|██████████| 10/10 [29:09<00:00, 174.92s/it]

Epoch 9
 train loss: 1.5295634258251924, train acc: 0.37449920177459717, train f1: 0.34138965606689453
 val loss: 1.6213542891156143, val acc: 0.35802963376045227, val f1: 0.30885761976242065



# Модель 3

In [69]:
class CNNModel3(nn.Module):
    
    def __init__(self):
        super().__init__()   

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.05)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.MaxPool2d(2),
            #nn.Dropout2d(0.1)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.05)
        )

        self.gap = nn.AdaptiveAvgPool2d((8, 8))

        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 256),
            nn.SiLU(),
            #nn.Dropout(0.1),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [70]:
model3 = CNNModel3()
model3 = model3.float()
model3 = model3.to(device)

In [71]:
loss_fn3 = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=0.1
)

In [72]:
optimizer = torch.optim.AdamW(model3.parameters(), lr=5e-4, weight_decay=1e-4)

In [73]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model3,
    optimizer,
    10,
    train_loader, 
    val_loader, 
    loss_fn
)

 10%|█         | 1/10 [03:00<27:02, 180.25s/it]

Epoch 0
 train loss: 1.754066869807549, train acc: 0.28485578298568726, train f1: 0.22824056446552277
 val loss: 1.743930300329901, val acc: 0.22787344455718994, val f1: 0.15202957391738892



 20%|██        | 2/10 [06:01<24:06, 180.79s/it]

Epoch 1
 train loss: 1.7083823207097175, train acc: 0.30629006028175354, train f1: 0.2513425946235657
 val loss: 1.6925261119368729, val acc: 0.36964356899261475, val f1: 0.2652983069419861



 30%|███       | 3/10 [08:55<20:42, 177.55s/it]

...

Epoch 7
 train loss: 1.5292969250526183, train acc: 0.3658854067325592, train f1: 0.3328206241130829
 val loss: 1.6668700124048124, val acc: 0.3404085040092468, val f1: 0.2968061864376068



 90%|█████████ | 9/10 [26:24<02:56, 176.67s/it]

Epoch 8
 train loss: 1.49221550529966, train acc: 0.3806089758872986, train f1: 0.347797691822052
 val loss: 1.7221605713200416, val acc: 0.3528233766555786, val f1: 0.2887446880340576



100%|██████████| 10/10 [29:24<00:00, 176.49s/it]

Epoch 9
 train loss: 1.4407801054036007, train acc: 0.4034455120563507, train f1: 0.37245696783065796
 val loss: 1.7179661153987715, val acc: 0.3536243438720703, val f1: 0.2988343834877014



# Модель 4

In [85]:
class CNNModel4(nn.Module):
    
    def __init__(self):
        super().__init__()   

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.02)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.MaxPool2d(2),
            #nn.Dropout2d(0.1)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.02)
        )

        self.gap = nn.AdaptiveAvgPool2d((6, 6))

        self.fc = nn.Sequential(
            nn.Linear(64 * 6 * 6, 256),
            nn.SiLU(),
            #nn.Dropout(0.1),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [86]:
model4 = CNNModel4()
model4 = model4.float()
model4 = model4.to(device)

In [87]:
optimizer = torch.optim.AdamW(
    model4.parameters(),
    lr=3e-4,
    weight_decay=1e-5
)

In [88]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=25
)

In [89]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model4,
    optimizer,
    25,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

  4%|▍         | 1/25 [02:52<1:08:54, 172.28s/it]

Epoch 0
 train loss: 1.7419294065389879, train acc: 0.29196715354919434, train f1: 0.2306133508682251
 val loss: 1.7188724241438944, val acc: 0.22947536408901215, val f1: 0.1906062513589859



  8%|▊         | 2/25 [05:39<1:04:57, 169.45s/it]

Epoch 1
 train loss: 1.6924621915588012, train acc: 0.3190104067325592, train f1: 0.26458626985549927
 val loss: 1.7165054318251882, val acc: 0.2647176682949066, val f1: 0.21124061942100525



 12%|█▏        | 3/25 [08:29<1:02:15, 169.78s/it]

...

Epoch 22
 train loss: 0.8788912430501137, train acc: 0.6366186141967773, train f1: 0.6299821138381958
 val loss: 1.889556767027469, val acc: 0.38325992226600647, val f1: 0.32601198554039



 96%|█████████▌| 24/25 [1:09:50<02:55, 175.04s/it]

Epoch 23
 train loss: 0.8714645717484064, train acc: 0.6424278616905212, train f1: 0.639160692691803
 val loss: 1.8824224355304318, val acc: 0.3840608596801758, val f1: 0.3289978504180908



100%|██████████| 25/25 [1:12:39<00:00, 174.37s/it]

Epoch 24
 train loss: 0.8695443318440363, train acc: 0.641526460647583, train f1: 0.6366502046585083
 val loss: 1.865244263866145, val acc: 0.38446134328842163, val f1: 0.32850372791290283

